In [ ]:
# ============================================================
# FAST QLoRA MEDICAL FINE-TUNING (FREE COLAB T4 FRIENDLY)
# ~25-40 MINUTES
# ============================================================

!pip -q install -U transformers datasets accelerate peft trl bitsandbytes

# ============================================================
# IMPORTS
# ============================================================

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# ============================================================
# DATASET
# ============================================================

print("Loading dataset...")

dataset = load_dataset(
    "lavita/ChatDoctor-HealthCareMagic-100k",
    split="train"
)

SYSTEM_PROMPT = """
You are a helpful medical assistant.

Provide educational medical information.
Do not claim to be a licensed physician.
Recommend consulting a healthcare professional for diagnosis.
Recommend urgent medical care for emergencies.
"""

def format_example(example):

    question = str(example.get("input", "")).strip()
    answer = str(example.get("output", "")).strip()

    answer = answer.replace("Chat Doctor", "")
    answer = answer.replace("Costco Chat Doctor", "")

    text = f"""### System:
{SYSTEM_PROMPT}

### User:
{question}

### Assistant:
{answer}
"""

    return {"text": text}

dataset = dataset.map(
    format_example,
    remove_columns=dataset.column_names
)

# ============================================================
# SMALLER DATASET = FASTER TRAINING
# ============================================================

dataset = dataset.shuffle(seed=42)

# Change to 5000 if you want slightly better results
dataset = dataset.select(range(3000))

split = dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_dataset = split["train"]
eval_dataset = split["test"]

print("Train samples:", len(train_dataset))
print("Eval samples :", len(eval_dataset))

# ============================================================
# MODEL
# ============================================================

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading model...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model.config.use_cache = False

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ============================================================
# LORA
# ============================================================

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

# ============================================================
# TRAINING CONFIG
# ============================================================

training_args = SFTConfig(
    output_dir="./medical_qlora",

    num_train_epochs=1,

    learning_rate=2e-4,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=2,

    logging_steps=20,

    save_strategy="epoch",

    bf16=torch.cuda.is_available(),

    packing=False,
)

# ============================================================
# TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
)

# ============================================================
# TRAIN
# ============================================================

print("Starting training...")

trainer.train()

print("Training complete!")

# ============================================================
# SAVE
# ============================================================

trainer.save_model("./medical_adapter")
tokenizer.save_pretrained("./medical_adapter")

print("Adapter saved!")

# ============================================================
# INFERENCE
# ============================================================

model = trainer.model
model.eval()

prompt = """
I have a sore throat and mild fever for 2 days.
What should I do?
"""

messages = [
    {
        "role": "system",
        "content": SYSTEM_PROMPT
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\n========================")
print("MODEL RESPONSE")
print("========================\n")
print(response)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.9 MB/s eta 0:00:00
Loading dataset...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

...

Starting training...


Step,Training Loss
20,2.813609
40,2.454935
60,2.291986
80,2.263249
100,2.290132
120,2.322928
140,2.242170
160,2.235955
180,2.225525
200,2.257223


Training complete!
Adapter saved!


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



MODEL RESPONSE

system

You are a helpful medical assistant.

Provide educational medical information.
Do not claim to be a licensed physician.
Recommend consulting a healthcare professional for diagnosis.
Recommend urgent medical care for emergencies.

user

I have a sore throat and mild fever for 2 days.
What should I do?

assistant
Hi, Welcome to Chat Doctor. Thank you for writing in . The common cause of sore throat is viral infection. This can cause fever too. So it's better if you take plenty of fluids, gargle with lukewarm saline water or warm water. Take anti-inflammatory tablet like ibuprofen. You may consult your doctor if the condition worsens. Hope this answers your query. If you have any further questions then please feel free to contact us again. Wish you good health. Take Care.

